# IMLE-Net: Interpretable Multi-level Multi-channel ECG Classification
**Paper**: IMLE-Net: An Interpretable Multi-level Multi-channel Model for ECG Classification (IEEE SMC 2021)  
**GitHub tác giả**: https://github.com/likith012/IMLE-Net  

**Pipeline**:
1. Mount Google Drive → load data
2. Preprocessing (đúng theo paper)
3. Build IMLE-Net (TensorFlow/Keras)
4. Train + save log mỗi lần train
5. Evaluate + Visualize
6. Attention visualization
7. Inference demo

## 1. Setup (chỉ cần chạy lần đầu)

In [1]:
# ── Kaggle: không cần mount drive, nhưng cần kết nối Google Drive qua API
# Nếu chạy trên Kaggle: upload data lên Kaggle Dataset hoặc dùng gdown
# Nếu chạy trên Colab: bỏ comment dòng dưới
# from google.colab import drive; drive.mount('/content/drive')

# ── Cài thư viện (Kaggle đã có sẵn hầu hết)
import subprocess, sys
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# gdown để download từ Google Drive
pip_install('gdown')
print('Setup done.')

Setup done.


In [2]:
# ── Import
import os, json, warnings, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # headless trên Kaggle
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    classification_report
)

print(f'TF  : {tf.__version__}')
print(f'GPU : {tf.config.list_physical_devices("GPU")}')

2026-04-14 16:22:39.130723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776183759.634217      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776183759.739905      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776183760.830067      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776183760.830112      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776183760.830115      55 computation_placer.cc:177] computation placer alr

TF  : 2.19.0
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 2. Configuration

In [3]:
# ================================================================
# CONFIGURATION  –  chỉnh sửa ở đây
# ================================================================

# ── Đường dẫn Google Drive (dùng gdown)
SIGNALS_GDRIVE_ID = 'YOUR_SIGNALS_FILE_ID'   # ← THAY VÀO ĐÂY
LABELS_GDRIVE_ID  = 'YOUR_LABELS_FILE_ID'    # ← THAY VÀO ĐÂY

USE_LOCAL        = True
LOCAL_SIGNALS    = '/kaggle/input/datasets/trnduynguyn/constructed-ptbxl-2/reconstructed_PTBXL_2/reconstructed_signals_2.csv'
LOCAL_LABELS     = '/kaggle/input/datasets/trnduynguyn/constructed-ptbxl-2/reconstructed_PTBXL_2/labels_2.csv'

OUTPUT_ROOT = '/kaggle/working/IMLE_Net_outputs'

# ── Hyperparameters (theo paper IMLE-Net IEEE SMC 2021 + code gốc tác giả)
# DATA: PTB-XL 250 Hz × 10s = 2500 samples/lead
# num_blocks_list=[2,2,2,2] theo code gốc → CNN output = 256 filters
CFG = {
    # Signal (PTB-XL 250 Hz, 10 sec)
    'num_channels' : 12,      # 12-lead ECG
    'signal_len'   : 2500,    # 10s × 250Hz
    'beat_len'     : 250,     # 1 beat = 1s = 250 samples ← dùng trong model reshape
    # num_beats = signal_len / beat_len = 10  (computed automatically)

    # Model (theo code gốc tác giả configs/imle_config.py)
    'start_filters'    : 32,          # Conv1D đầu tiên
    'num_blocks_list'  : [2, 2, 2, 2],# 4 stages: 32→64→128→256 filters
    'kernel_size'      : 8,           # kernel size residual block
    'lstm_units'       : 64,          # BiLSTM units (output 128 sau bidirectional)
    'attention_dim'    : 64,          # attention hidden dim

    # Training (theo paper)
    'batch_size'   : 32,
    'epochs'       : 60,
    'lr'           : 1e-3,
    'val_split'    : 0.15,
    'test_split'   : 0.15,
    'seed'         : 42,
    'patience'     : 20,     # theo paper EarlyStopping patience=20
    'threshold'    : 0.5,
}
CFG['num_beats'] = CFG['signal_len'] // CFG['beat_len']  # = 10

# ── Run ID & directory
from datetime import datetime
RUN_TS  = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = os.path.join(OUTPUT_ROOT, f'run_{RUN_TS}')
os.makedirs(RUN_DIR, exist_ok=True)
print(f'Run directory: {RUN_DIR}')

with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump(CFG, f, indent=4)

tf.random.set_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('Config saved.')


Run directory: /kaggle/working/IMLE_Net_outputs/run_20260414_162315
Config saved.


## 3. Download / Load Data

In [4]:
# ================================================================
# DOWNLOAD từ Google Drive (nếu không dùng local)
# ================================================================
import gdown

if USE_LOCAL:
    SIGNALS_PATH = LOCAL_SIGNALS
    LABELS_PATH  = LOCAL_LABELS
    print('Using local files.')
else:
    SIGNALS_PATH = '/kaggle/working/reconstructed_signals.csv'
    LABELS_PATH  = '/kaggle/working/labels.csv'

    if not os.path.exists(SIGNALS_PATH):
        print('Downloading reconstructed_signals.csv ...')
        gdown.download(id=SIGNALS_GDRIVE_ID, output=SIGNALS_PATH, quiet=False)
    else:
        print('Signals file already exists.')

    if not os.path.exists(LABELS_PATH):
        print('Downloading labels.csv ...')
        gdown.download(id=LABELS_GDRIVE_ID, output=LABELS_PATH, quiet=False)
    else:
        print('Labels file already exists.')

print(f'Signals: {SIGNALS_PATH}')
print(f'Labels : {LABELS_PATH}')

Using local files.
Signals: /kaggle/input/datasets/trnduynguyn/constructed-ptbxl-2/reconstructed_PTBXL_2/reconstructed_signals_2.csv
Labels : /kaggle/input/datasets/trnduynguyn/constructed-ptbxl-2/reconstructed_PTBXL_2/labels_2.csv


## 4. Preprocessing (theo paper)

In [5]:
# ================================================================
# LOAD & PARSE DATA
# ================================================================
# Cấu trúc reconstructed_signals.csv:
#   sample_id, element_id, I_groundtruth, I_predicted, II_groundtruth, II_predicted, ...
#   Mỗi hàng là 1 time-step của 1 bệnh nhân (element_id)
#   → Cần group theo element_id, lấy các cột *_groundtruth
#
# 12 leads theo thứ tự chuẩn:
#   I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

print('Loading signals.csv ...')
sig_df = pd.read_csv(SIGNALS_PATH)
print(f'  Raw shape: {sig_df.shape}')
print(f'  Columns  : {list(sig_df.columns[:8])} ...')

print('\nLoading labels.csv ...')
lab_df = pd.read_csv(LABELS_PATH)
print(f'  Shape    : {lab_df.shape}')
print(f'  Columns  : {list(lab_df.columns)}')
print(lab_df.head(3))

Loading signals.csv ...
  Raw shape: (9400, 27)
  Columns  : ['sample_id', 'element_id', 'subset', 'I_groundtruth', 'I_predicted', 'II_groundtruth', 'II_predicted', 'III_groundtruth'] ...

Loading labels.csv ...
  Shape    : (9400, 7)
  Columns  : ['element_id', 'NORM', 'MI', 'STTC', 'CD', 'HYP', 'subset']
  element_id  NORM  MI  STTC  CD  HYP subset
0    ptbxl_4     1   0     0   0    0  train
1    ptbxl_7     1   0     0   0    0  train
2    ptbxl_8     0   1     0   0    0  train


In [6]:
# ================================================================
# BUILD SIGNAL ARRAY  (N, 12, 1000)
# ================================================================
# Xác định cột groundtruth cho từng lead
gt_cols = {}
for lead in LEAD_NAMES:
    col = f'{lead}_groundtruth'
    if col in sig_df.columns:
        gt_cols[lead] = col
    else:
        # Thử predicted nếu không có groundtruth
        col2 = f'{lead}_predicted'
        if col2 in sig_df.columns:
            gt_cols[lead] = col2
            print(f'  Warning: using predicted for {lead}')
        else:
            print(f'  ERROR: cannot find column for lead {lead}')

print(f'Using columns: { {k:v for k,v in list(gt_cols.items())} }')

# Xác định cột element_id
id_col = 'element_id' if 'element_id' in sig_df.columns else sig_df.columns[0]
print(f'ID column: {id_col}')

Using columns: {'I': 'I_groundtruth', 'II': 'II_groundtruth', 'III': 'III_groundtruth', 'aVR': 'aVR_groundtruth', 'aVL': 'aVL_groundtruth', 'aVF': 'aVF_groundtruth', 'V1': 'V1_groundtruth', 'V2': 'V2_groundtruth', 'V3': 'V3_groundtruth', 'V4': 'V4_groundtruth', 'V5': 'V5_groundtruth', 'V6': 'V6_groundtruth'}
ID column: element_id


In [7]:
# ── Group theo element_id, lấy signal_len time-steps đầu tiên
print('Building signal array ...')

# Lấy các element_id có trong cả hai file
label_ids = set(lab_df['element_id'].astype(str).tolist())
sig_ids   = set(sig_df[id_col].astype(str).tolist())
common_ids = sorted(label_ids & sig_ids)
print(f'  Common element_ids: {len(common_ids)}')

SIGNAL_LEN = CFG['signal_len']
N          = len(common_ids)
C          = len(LEAD_NAMES)

X_raw = np.zeros((N, C, SIGNAL_LEN), dtype=np.float32)
sig_df['_id_str'] = sig_df[id_col].astype(str)
grouped = sig_df.groupby('_id_str')
def parse_signal_cell(cell_value):
    """
    Mỗi cell chứa toàn bộ time-series dưới dạng string '0.29 0.28 ...'
    hoặc scalar float. Trả về numpy array float32.
    """
    if isinstance(cell_value, str):
        return np.array(cell_value.split(), dtype=np.float32)
    else:
        return np.array([cell_value], dtype=np.float32)

for i, eid in enumerate(tqdm(common_ids, desc='Parsing signals')):
    grp = grouped.get_group(eid)
    if 'sample_id' in grp.columns:
        grp = grp.sort_values('sample_id')

    for j, lead in enumerate(LEAD_NAMES):
        raw_vals = grp[gt_cols[lead]].values
        parsed = np.concatenate([parse_signal_cell(v) for v in raw_vals])
        T = min(len(parsed), SIGNAL_LEN)
        X_raw[i, j, :T] = parsed[:T]
        if T < SIGNAL_LEN:
            X_raw[i, j, T:] = parsed[-1] if T > 0 else 0.0

print(f'X_raw shape: {X_raw.shape}  (N, C, T)')

# Lưu thứ tự element_id
id_list = np.array(common_ids)
del sig_df, grouped; gc.collect()


Building signal array ...
  Common element_ids: 9400


Parsing signals:   0%|          | 0/9400 [00:00<?, ?it/s]

X_raw shape: (9400, 12, 2500)  (N, C, T)


49

In [8]:
# ================================================================
# LABELS  (N, num_classes)
# ================================================================
# labels.csv: element_id, NORM, MI, STTC, CD, HYP  (one-hot)
CLASS_NAMES = [c for c in lab_df.columns if c != 'element_id']
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES == 5, f'Expected 5 classes, got {NUM_CLASSES}: {CLASS_NAMES}'
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

lab_df['_id_str'] = lab_df['element_id'].astype(str)
lab_indexed = lab_df.set_index('_id_str')[CLASS_NAMES]

Y_raw = np.zeros((N, NUM_CLASSES), dtype=np.float32)
for i, eid in enumerate(common_ids):
    Y_raw[i] = lab_indexed.loc[eid].values.astype(np.float32)

print(f'Y_raw shape: {Y_raw.shape}')
print('Class distribution:')
for cls, cnt in zip(CLASS_NAMES, Y_raw.sum(axis=0)):
    print(f'  {cls:<8}: {int(cnt):5d}  ({cnt/N*100:.1f}%)')

AssertionError: Expected 5 classes, got 6: ['NORM', 'MI', 'STTC', 'CD', 'HYP', 'subset']

In [ ]:
# ================================================================
# NORMALIZE (theo paper: StandardScaler fit trên tập train)
# ================================================================
# Paper dùng sklearn StandardScaler fit toàn bộ train set:
#   scaler.fit(np.vstack(X_train).flatten()[:, np.newaxis])
# → Global scaler (không phải per-sample normalization)
# Tại bước này chưa split → fit scaler sau khi có X_train (xem cell split)
#
# Để đúng thứ tự: lưu X_raw & Y_raw, split trước, rồi fit scaler trên train.
# Cell này chỉ in thông tin, scaler sẽ được fit ở cell split bên dưới.
print(f'X_raw shape : {X_raw.shape}  (N, C, T)')
print(f'Y_raw shape : {Y_raw.shape}')
print('Scaler sẽ được fit sau khi split train/val/test.')


In [ ]:
# ================================================================
# NOTE: Beat segmentation KHÔNG làm trước — làm bên trong model
# ================================================================
# Theo paper IMLE-Net: model nhận input (batch, channels, signal_len, 1)
# rồi dùng tf.reshape bên trong để tách beats:
#   reshape(-1, beat_len, 1)  → áp CNN
#   reshape(-1, num_beats, 128) → áp BiLSTM
#   reshape(-1, num_channels, 128) → channel attention
#
# Input shape cuối cùng: (N, num_channels, signal_len, 1)
# KHÔNG phải (N, num_channels, num_beats, beat_len)
print('Beat segmentation: xử lý bên trong model bằng tf.reshape.')
print(f'Input shape sẽ là: (N, {CFG["num_channels"]}, {CFG["signal_len"]}, 1)')


In [ ]:
# ================================================================
# TRAIN / VAL / TEST SPLIT + STANDARDSCALER (theo paper)
# ================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle as sk_shuffle

idx = np.arange(N)
strat_label = Y_raw.argmax(axis=1)

idx_tv, idx_test = train_test_split(
    idx, test_size=CFG['test_split'], random_state=CFG['seed'],
    stratify=strat_label
)
idx_train, idx_val = train_test_split(
    idx_tv,
    test_size=CFG['val_split'] / (1 - CFG['test_split']),
    random_state=CFG['seed'], stratify=strat_label[idx_tv]
)

X_train_raw, Y_train = X_raw[idx_train], Y_raw[idx_train]
X_val_raw,   Y_val   = X_raw[idx_val],   Y_raw[idx_val]
X_test_raw,  Y_test  = X_raw[idx_test],  Y_raw[idx_test]

# ── Fit StandardScaler trên tập train (đúng theo preprocess.py gốc)
# np.vstack: (N_train*C, T) → flatten → (N_train*C*T, 1)
scaler = StandardScaler()
scaler.fit(np.vstack(X_train_raw).flatten()[:, np.newaxis].astype(float))

def apply_scaler(inputs, scaler):
    """Apply scaler to each sample independently (shape: N, C, T)."""
    temp = []
    for x in inputs:
        x_shape = x.shape
        temp.append(scaler.transform(x.flatten()[:, np.newaxis]).reshape(x_shape))
    return np.array(temp, dtype=np.float32)

print('Applying StandardScaler ...')
X_train_scale = apply_scaler(X_train_raw, scaler)
X_val_scale   = apply_scaler(X_val_raw,   scaler)
X_test_scale  = apply_scaler(X_test_raw,  scaler)

# Shuffle train (theo paper)
X_train_scale, Y_train = sk_shuffle(X_train_scale, Y_train, random_state=CFG['seed'])

# ── Thêm channel cuối (axis -1) → (N, C, T, 1) cho model
X_train = X_train_scale[..., np.newaxis]   # (N, 12, 1000, 1)
X_val   = X_val_scale[..., np.newaxis]
X_test  = X_test_scale[..., np.newaxis]

# Giữ X_norm (C,T) không có axis cuối cho visualization
X_norm = X_train_scale   # dùng cho EDA plot

print(f'Train : {X_train.shape}  Labels: {Y_train.shape}')
print(f'Val   : {X_val.shape}  Labels: {Y_val.shape}')
print(f'Test  : {X_test.shape}  Labels: {Y_test.shape}')


In [ ]:
# ================================================================
# EDA VISUALIZATION
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Class distribution
counts = Y_raw.sum(axis=0)
colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
axes[0].bar(CLASS_NAMES, counts, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Count')
for i, (c, cnt) in enumerate(zip(CLASS_NAMES, counts)):
    axes[0].text(i, cnt + 5, str(int(cnt)), ha='center', fontsize=9)

# Sample ECG (first sample, all 12 leads)
t = np.arange(CFG['signal_len'])
offset = 5
for c in range(CFG['num_channels']):
    sig = X_norm[0, c]
    axes[1].plot(t, sig + c * offset, linewidth=0.8, alpha=0.9, label=LEAD_NAMES[c])
axes[1].set_title('Sample ECG – All 12 Leads', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Time (samples at 100 Hz)')
axes[1].set_yticks([c * offset for c in range(CFG['num_channels'])])
axes[1].set_yticklabels(LEAD_NAMES)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
eda_path = os.path.join(RUN_DIR, 'eda.png')
plt.savefig(eda_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'EDA saved → {eda_path}')

## 5. IMLE-Net Architecture (TensorFlow/Keras)

In [ ]:
# ================================================================
# ATTENTION LAYER — copy chính xác từ code gốc tác giả (IMLENet.py)
# ================================================================
class attention(layers.Layer):
    """
    Feed-forward attention — CHÍNH XÁC theo code gốc tác giả likith012.
    alpha = softmax(V.T * tanh(W.T * x + b))
    Input : (batch, seq_len, features)
    Output: context_vector (batch, features), attention_weights (batch, seq_len, 1)
    """
    def __init__(self, return_sequences=False, dim=64, **kwargs):
        self.return_sequences = return_sequences
        self.dim = dim
        super(attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='att_weight', shape=(input_shape[-1], self.dim), initializer='normal'
        )
        self.b = self.add_weight(
            name='att_bias', shape=(input_shape[1], self.dim), initializer='zeros'
        )
        self.V = self.add_weight(
            name='Vatt', shape=(self.dim, 1), initializer='normal'
        )
        super(attention, self).build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        e = K.dot(e, self.V)
        a = K.softmax(e, axis=1)
        output = x * a
        if self.return_sequences:
            return output, a
        return K.sum(output, axis=1), a

    def get_config(self):
        base_config = super().get_config()
        config = {'return_sequences': self.return_sequences, 'dim': self.dim}
        return dict(list(base_config.items()) + list(config.items()))

print('attention layer (author-exact) defined.')


In [ ]:
# ================================================================
# BUILD IMLE-Net — copy chính xác logic từ code gốc tác giả (IMLENet.py)
# với sửa đúng beat_att_dim thay vì hardcode 128
# ================================================================

def relu_bn(inputs):
    """ReLU → BatchNorm — đúng theo code gốc tác giả."""
    from tensorflow.keras.layers import ReLU, BatchNormalization
    relu = ReLU()(inputs)
    bn = BatchNormalization()(relu)
    return bn

def residual_block(x, downsample, filters, kernel_size=8):
    """
    Residual block — CHÍNH XÁC theo code gốc tác giả.
    Shortcut: Conv1x1 stride=2 khi downsample, identity khi không.
    """
    y = layers.Conv1D(
        kernel_size=kernel_size,
        strides=(1 if not downsample else 2),
        filters=filters,
        padding='same',
    )(x)
    y = relu_bn(y)
    y = layers.Conv1D(kernel_size=kernel_size, strides=1, filters=filters, padding='same')(y)
    if downsample:
        x = layers.Conv1D(kernel_size=1, strides=2, filters=filters, padding='same')(x)
    out = layers.Add()([x, y])
    out = relu_bn(out)
    return out


def build_imle_net(num_channels, signal_len, beat_len, num_classes,
                   start_filters, num_blocks_list, kernel_size,
                   lstm_units, attention_dim):
    """
    Xây dựng IMLE-Net đúng theo code gốc tác giả.
    num_blocks_list=[2,2,2,2] → 4 stages: 32→64→128→256 filters

    Shape flow (signal_len=2500, beat_len=250, num_channels=12):
      beat_reshape  : (B,12,2500,1) → (B×120, 250, 1)
      CNN 4 stages  : (B×120, 250, 1) → (B×120, 32) → ... → (B×120, ~31, 256)
      beat_att      : (B×120, ~31, 256) → (B×120, 256)   [beat_att_dim=256]
      rhythm_reshape: (B×120, 256)     → (B×12, 10, 256)
      BiLSTM(64)    : (B×12, 10, 256)  → (B×12, 10, 128)
      rhythm_att    : (B×12, 10, 128)  → (B×12, 128)
      channel_reshape:(B×12, 128)      → (B, 12, 128)
      channel_att   : (B, 12, 128)     → (B, 128)
      Dense         : (B, 128)         → (B, num_classes)
    """
    num_beats = signal_len // beat_len   # 2500//250 = 10

    # beat_att_dim = filters output CNN stage cuối
    # [2,2,2,2] → stage 0:32, 1:64, 2:128, 3:256 → beat_att_dim = 256
    beat_att_dim = start_filters * (2 ** (len(num_blocks_list) - 1))
    bilstm_dim   = lstm_units * 2   # BiLSTM(64) → 128

    inputs = keras.Input(shape=(num_channels, signal_len, 1), name='ecg_input')

    # ── Beat Level ────────────────────────────────────────────────
    # (B,12,2500,1) → (B×120, 250, 1)
    x = layers.Lambda(
        lambda t: tf.reshape(t, (-1, beat_len, 1)),
        name='beat_reshape'
    )(inputs)

    # Initial Conv (đúng theo code gốc)
    x = layers.Conv1D(filters=start_filters, kernel_size=kernel_size, padding='same')(x)
    x = layers.Activation('relu')(x)

    # Residual blocks — đúng theo code gốc
    num_filters = start_filters
    for i in range(len(num_blocks_list)):
        num_blocks = num_blocks_list[i]
        for j in range(num_blocks):
            x = residual_block(x, downsample=(j == 0 and i != 0), filters=num_filters, kernel_size=kernel_size)
        num_filters *= 2
    # Sau vòng lặp: x = (B×120, T', beat_att_dim)  [beat_att_dim=256 với [2,2,2,2]]

    # Beat Attention → (B×120, beat_att_dim)
    x, _ = attention(name='beat_att')(x)

    # ── Rhythm Level ──────────────────────────────────────────────
    # FIX KEY: dùng beat_att_dim (256) thay vì hardcode 128
    # (B×120, 256) → (B×12, 10, 256)
    rhythm_len = num_beats   # = 10
    x = layers.Lambda(
        lambda t: tf.reshape(t, (-1, rhythm_len, beat_att_dim)),
        name='rhythm_reshape'
    )(x)

    # BiLSTM(64) → (B×12, 10, 128)
    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True),
        name='bilstm'
    )(x)

    # Rhythm Attention → (B×12, 128)
    x, _ = attention(name='rhythm_att')(x)

    # ── Channel Level ─────────────────────────────────────────────
    # (B×12, 128) → (B, 12, 128)
    x = layers.Lambda(
        lambda t: tf.reshape(t, (-1, num_channels, bilstm_dim)),
        name='channel_reshape'
    )(x)

    # Channel Attention → (B, 128)
    x, _ = attention(name='channel_att')(x)

    # ── Output ────────────────────────────────────────────────────
    outputs = layers.Dense(num_classes, activation='sigmoid', name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='IMLE_Net')
    return model


# ── Build
model = build_imle_net(
    num_channels    = CFG['num_channels'],
    signal_len      = CFG['signal_len'],
    beat_len        = CFG['beat_len'],
    num_classes     = NUM_CLASSES,
    start_filters   = CFG['start_filters'],
    num_blocks_list = CFG['num_blocks_list'],
    kernel_size     = CFG['kernel_size'],
    lstm_units      = CFG['lstm_units'],
    attention_dim   = CFG['attention_dim'],
)

model.summary()
print(f'\nTotal params: {model.count_params():,}')


## 6. Training

In [ ]:
# ================================================================
# COMPILE (theo paper)
# ================================================================
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=CFG['lr']),
    loss      = keras.losses.BinaryCrossentropy(),
    metrics   = [
        'accuracy',
        keras.metrics.AUC(name='auc', multi_label=True),
    ]
)

# ── Callbacks (theo paper)
ckpt_path = os.path.join(RUN_DIR, 'best_model.weights.h5')  # .h5 để load_weights
csv_log   = os.path.join(RUN_DIR, 'training_log.csv')

# Custom checkpoint (lưu best val_auc sau epoch 5, theo callbacks.py gốc)
class BestAUCCheckpoint(keras.callbacks.Callback):
    def __init__(self, filepath, monitor='val_auc', start_epoch=5):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.start_epoch = start_epoch
        self.best_score = 0.0

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor, 0)
        if epoch >= self.start_epoch and current > self.best_score:
            self.best_score = current
            self.model.save_weights(self.filepath)
            print(f'  → Saved best weights (epoch {epoch+1}, {self.monitor}={current:.4f})')

callbacks = [
    BestAUCCheckpoint(ckpt_path, monitor='val_auc', start_epoch=5),
    keras.callbacks.EarlyStopping(
        monitor='val_auc', mode='max',
        min_delta=0.0001, patience=CFG['patience'],
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-6, verbose=1
    ),
    keras.callbacks.CSVLogger(csv_log, append=True),
]

print(f'Checkpoint  : {ckpt_path}')
print(f'Training log: {csv_log}')
print(f'Train samples: {len(X_train)}')
print(f'Val   samples: {len(X_val)}')
print(f'Batch size   : {CFG["batch_size"]}')
print(f'Epochs       : {CFG["epochs"]}')


In [ ]:
# ================================================================
# TRAIN
# ================================================================
# Kiểm tra shape trước khi train
assert X_train.shape[1:] == (CFG['num_channels'], CFG['signal_len'], 1), \
    f'X shape sai: {X_train.shape}'
assert Y_train.shape[1] == NUM_CLASSES, \
    f'Y shape sai: {Y_train.shape}'
assert model.output_shape == (None, NUM_CLASSES), \
    f'Model output sai: {model.output_shape}'
print(f'Shape check OK — X:{X_train.shape}, Y:{Y_train.shape}, model:{model.output_shape}')

history = model.fit(
    X_train, Y_train,
    validation_data = (X_val, Y_val),
    epochs          = CFG['epochs'],
    batch_size      = CFG['batch_size'],
    callbacks       = callbacks,
    verbose         = 1,
)
print('Training done.')

## 7. Evaluation & Visualization

In [ ]:
# ================================================================
# TRAINING CURVES
# ================================================================
hist = history.history
metric_pairs = [
    ('loss',      'Loss'),
    ('accuracy',  'Accuracy'),
    ('auc',       'AUC'),
    ('precision', 'Precision'),
    ('recall',    'Recall'),
]

ncols = 3
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 10))
axes = axes.flatten()

for i, (m, title) in enumerate(metric_pairs):
    if m in hist:
        axes[i].plot(hist[m],          label='Train', linewidth=2)
        axes[i].plot(hist[f'val_{m}'], label='Val',   linewidth=2, linestyle='--')
        axes[i].set_title(title, fontweight='bold', fontsize=12)
        axes[i].set_xlabel('Epoch')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

# LR curve
if 'lr' in hist:
    axes[5].plot(hist['lr'], color='darkorange', linewidth=2)
    axes[5].set_title('Learning Rate', fontweight='bold', fontsize=12)
    axes[5].set_xlabel('Epoch')
    axes[5].set_yscale('log')
    axes[5].grid(True, alpha=0.3)

plt.suptitle(f'IMLE-Net Training Curves  |  Run: {RUN_TS}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
curves_path = os.path.join(RUN_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved → {curves_path}')

In [ ]:
# ================================================================
# TEST SET EVALUATION
# ================================================================
Y_prob = model.predict(X_test, batch_size=CFG['batch_size'], verbose=1)
Y_pred = (Y_prob >= CFG['threshold']).astype(int)

# Per-class AUC
auc_per_class = {}
for i, cls in enumerate(CLASS_NAMES):
    try:
        auc_per_class[cls] = round(roc_auc_score(Y_test[:, i], Y_prob[:, i]), 4)
    except:
        auc_per_class[cls] = None

auc_macro = round(roc_auc_score(Y_test, Y_prob, average='macro'),  4)
auc_micro = round(roc_auc_score(Y_test, Y_prob, average='micro'),  4)
f1_macro  = round(f1_score(Y_test, Y_pred, average='macro',  zero_division=0), 4)
f1_micro  = round(f1_score(Y_test, Y_pred, average='micro',  zero_division=0), 4)
acc       = round(accuracy_score(Y_test, Y_pred), 4)

results = {
    'run_id'        : RUN_TS,
    'test_samples'  : len(X_test),
    'auc_macro'     : auc_macro,
    'auc_micro'     : auc_micro,
    'f1_macro'      : f1_macro,
    'f1_micro'      : f1_micro,
    'accuracy'      : acc,
    'best_val_auc'  : round(max(hist['val_auc']), 4),
    'best_epoch'    : int(np.argmax(hist['val_auc'])) + 1,
    'auc_per_class' : auc_per_class,
}

print('\n========== TEST RESULTS ==========')
for k, v in results.items():
    if k != 'auc_per_class':
        print(f'  {k:<20}: {v}')
print('  Per-class AUC:')
for cls, auc in auc_per_class.items():
    print(f'    {cls:<8}: {auc}')
print('==================================')

res_path = os.path.join(RUN_DIR, 'test_results.json')
with open(res_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f'Results saved → {res_path}')

In [ ]:
# ================================================================
# ROC CURVES  (per class)
# ================================================================
ncols = min(NUM_CLASSES, 3)
nrows = (NUM_CLASSES + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten()

for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(Y_test[:, i], Y_prob[:, i])
    auc_val     = auc_per_class[cls]
    axes[i].plot(fpr, tpr, lw=2.5, color='steelblue', label=f'AUC = {auc_val:.3f}')
    axes[i].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
    axes[i].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
    axes[i].set_title(cls, fontweight='bold', fontsize=13)
    axes[i].set_xlabel('False Positive Rate')
    axes[i].set_ylabel('True Positive Rate')
    axes[i].legend(fontsize=10)
    axes[i].grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'ROC Curves per Class  |  macro-AUC={auc_macro}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
roc_path = os.path.join(RUN_DIR, 'roc_curves.png')
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC curves saved → {roc_path}')

In [ ]:
# ================================================================
# CONFUSION MATRICES  (per class)
# ================================================================
ncols = min(NUM_CLASSES, 3)
nrows = (NUM_CLASSES + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten()

for i, cls in enumerate(CLASS_NAMES):
    cm = confusion_matrix(Y_test[:, i], Y_pred[:, i])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
        xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'],
        linewidths=0.5, linecolor='grey'
    )
    axes[i].set_title(cls, fontweight='bold', fontsize=13)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(RUN_DIR, 'confusion_matrices.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrices saved → {cm_path}')

In [ ]:
# ================================================================
# PRECISION-RECALL CURVES
# ================================================================
ncols = min(NUM_CLASSES, 3)
nrows = (NUM_CLASSES + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten()

for i, cls in enumerate(CLASS_NAMES):
    prec, rec, _ = precision_recall_curve(Y_test[:, i], Y_prob[:, i])
    axes[i].plot(rec, prec, lw=2.5, color='darkorange')
    axes[i].fill_between(rec, prec, alpha=0.1, color='darkorange')
    axes[i].set_title(cls, fontweight='bold', fontsize=13)
    axes[i].set_xlabel('Recall')
    axes[i].set_ylabel('Precision')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Precision-Recall Curves per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
pr_path = os.path.join(RUN_DIR, 'pr_curves.png')
plt.savefig(pr_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'PR curves saved → {pr_path}')

In [ ]:
# ================================================================
# CLASSIFICATION REPORT
# ================================================================
report = classification_report(
    Y_test, Y_pred, target_names=CLASS_NAMES, zero_division=0, output_dict=True
)
report_df = pd.DataFrame(report).T.round(4)
report_df.to_csv(os.path.join(RUN_DIR, 'classification_report.csv'))
print(report_df)

## 8. Attention Visualization

In [ ]:
# ================================================================
# BUILD ATTENTION EXTRACTION MODEL (đúng tên layer)
# ================================================================
def build_attention_extractor(model):
    """
    Trích xuất attention weights tại 3 level:
    - beat_att   : beat-level attention
    - rhythm_att : rhythm-level attention
    - channel_att: channel-level attention  ← dùng để visualize
    """
    try:
        ch_att_output = model.get_layer('channel_att').output[1]  # (B, 12, 1)
        att_model = keras.Model(
            inputs  = model.input,
            outputs = [model.output, ch_att_output],
            name    = 'att_extractor'
        )
        return att_model
    except Exception as e:
        print(f'Cannot build attention extractor: {e}')
        return None

att_model = build_attention_extractor(model)
print('Attention extractor:', 'OK' if att_model else 'FAILED')


In [ ]:
# ================================================================
# CHANNEL ATTENTION VISUALIZATION
# ================================================================
if att_model is not None:
    sample_idx = 0
    # X_test shape: (N, 12, 1000, 1)
    sample_x   = X_test[sample_idx:sample_idx + 1]
    true_label = Y_test[sample_idx]
    true_diag  = [CLASS_NAMES[i] for i, v in enumerate(true_label) if v == 1]

    preds, ch_alpha = att_model.predict(sample_x, verbose=0)
    # ch_alpha shape: (B, 12, 1) → squeeze → (12,)
    ch_alpha = ch_alpha[0].squeeze()    # (12,)
    pred_label = (preds[0] >= CFG['threshold']).astype(int)
    pred_diag  = [CLASS_NAMES[i] for i, v in enumerate(pred_label) if v == 1]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ── Bar chart: channel attention weights
    colors = plt.cm.RdYlGn(ch_alpha / (ch_alpha.max() + 1e-8))
    bars = axes[0].bar(LEAD_NAMES, ch_alpha, color=colors, edgecolor='grey', linewidth=0.8)
    axes[0].set_title('Channel-Level Attention Weights', fontweight='bold', fontsize=13)
    axes[0].set_ylabel('Attention Weight')
    axes[0].set_xlabel('ECG Lead')
    for bar, val in zip(bars, ch_alpha):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    axes[0].tick_params(axis='x', rotation=45)

    # ── ECG signals colored by attention
    t          = np.arange(CFG['signal_len'])
    norm_alpha = (ch_alpha - ch_alpha.min()) / (ch_alpha.max() - ch_alpha.min() + 1e-8)
    cmap       = plt.cm.RdYlGn
    offset_step = 5

    # raw_sig từ X_test_scale: (N, C, T) → lấy sample idx_test[sample_idx]
    raw_sig = X_test_scale[sample_idx]   # (12, 1000)
    for c in range(CFG['num_channels']):
        sig = raw_sig[c]
        axes[1].plot(t, sig + c * offset_step,
                     color=cmap(norm_alpha[c]), linewidth=1.5, alpha=0.9)
        axes[1].text(CFG['signal_len'] + 10, c * offset_step,
                     LEAD_NAMES[c], fontsize=9, va='center')

    sm = plt.cm.ScalarMappable(cmap=cmap,
                                norm=plt.Normalize(norm_alpha.min(), norm_alpha.max()))
    plt.colorbar(sm, ax=axes[1], label='Attention (normalized)', shrink=0.8)
    axes[1].set_title(
        f'ECG Colored by Channel Attention\n'
        f'True: {true_diag}  |  Pred: {pred_diag}',
        fontweight='bold', fontsize=11
    )
    axes[1].set_xlabel('Sample (100 Hz)')
    axes[1].set_yticks([])
    axes[1].grid(True, alpha=0.1)

    plt.tight_layout()
    att_path = os.path.join(RUN_DIR, 'attention_viz.png')
    plt.savefig(att_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Attention viz saved → {att_path}')
else:
    print('Attention visualization skipped.')


## 9. Save Everything

In [ ]:
# ================================================================
# SAVE MODEL
# ================================================================
# Keras format (.keras) – tương thích TF 2.12+
final_model_path = os.path.join(RUN_DIR, 'imle_net_final.keras')
model.save(final_model_path)
print(f'Model saved → {final_model_path}')

# ================================================================
# SUMMARY LOG
# ================================================================
lines = [
    '=' * 50,
    'IMLE-Net Run Summary',
    '=' * 50,
    f'Run ID       : {RUN_TS}',
    f'Run Dir      : {RUN_DIR}',
    '',
    '── Data',
    f'  Total samples : {N}',
    f'  Train         : {len(X_train)}',
    f'  Val           : {len(X_val)}',
    f'  Test          : {len(X_test)}',
    f'  Classes       : {CLASS_NAMES}',
    '',
    '── Config',
]
for k, v in CFG.items():
    lines.append(f'  {k:<20}: {v}')
lines += [
    '',
    '── Test Results',
]
for k, v in results.items():
    if k != 'auc_per_class':
        lines.append(f'  {k:<20}: {v}')
lines += ['', '── Per-class AUC']
for cls, auc in auc_per_class.items():
    lines.append(f'  {cls:<8}: {auc}')

summary_text = '\n'.join(lines)
summary_path = os.path.join(RUN_DIR, 'summary.txt')
with open(summary_path, 'w') as f:
    f.write(summary_text)
print(summary_text)

# ================================================================
# LIST FILES IN RUN DIR
# ================================================================
print(f'\n── Files in {RUN_DIR}:')
for root, dirs, files in os.walk(RUN_DIR):
    level = root.replace(RUN_DIR, '').count(os.sep)
    indent = '  ' * level
    for file in files:
        fpath = os.path.join(root, file)
        fsize = os.path.getsize(fpath) / 1024
        print(f'{indent}{file}  ({fsize:.1f} KB)')

In [ ]:
# ================================================================
# UPLOAD OUTPUTS LÊN GOOGLE DRIVE (tùy chọn)
# ================================================================
# Cách 1: dùng gdown upload (cần oauth)
# Cách 2: zip output dir và download về máy

import shutil
zip_path = f'/kaggle/working/imle_net_run_{RUN_TS}'
shutil.make_archive(zip_path, 'zip', RUN_DIR)
print(f'ZIP created: {zip_path}.zip')
print('→ Download từ Kaggle Output tab, rồi upload lên Google Drive.')

# Nếu đang trên Colab + Drive đã mount:
# import shutil
# shutil.copytree(RUN_DIR, f'/content/drive/MyDrive/IMLE-Net-Project/outputs/run_{RUN_TS}')
# print('Copied to Google Drive.')

## 10. Inference Demo

In [ ]:
# ================================================================
# INFERENCE FUNCTION (đúng input format)
# ================================================================
def predict_ecg(signal_array, model, scaler, num_channels=12, signal_len=1000,
                threshold=0.5, class_names=None):
    """
    signal_array: (num_channels, signal_len) numpy float32 — raw signal
    scaler: StandardScaler đã fit trên tập train
    Returns: dict với proba và prediction per class
    """
    # Apply StandardScaler (giống apply_scaler trong paper)
    x_shape = signal_array.shape
    sig = scaler.transform(signal_array.flatten()[:, np.newaxis]).reshape(x_shape)
    sig = sig.astype(np.float32)

    # Thêm batch dim và channel dim: (12, 1000) → (1, 12, 1000, 1)
    x = sig[np.newaxis, ..., np.newaxis]   # (1, 12, 1000, 1)

    prob = model.predict(x, verbose=0)[0]  # (num_classes,)
    pred = (prob >= threshold).astype(int)

    out = {}
    for i, cls in enumerate(class_names or [str(i) for i in range(len(prob))]):
        out[cls] = {'prob': float(prob[i]), 'pred': int(pred[i])}
    return out


# ── Demo trên một mẫu test
# X_test_raw: (N, C, T) chưa normalize
sample_raw  = X_test_raw[0]   # (12, 1000)
result      = predict_ecg(
    sample_raw, model, scaler,
    num_channels = CFG['num_channels'],
    signal_len   = CFG['signal_len'],
    threshold    = CFG['threshold'],
    class_names  = CLASS_NAMES
)

print('\nInference demo:')
print(f'True labels: { [CLASS_NAMES[i] for i,v in enumerate(Y_test[0]) if v==1] }')
print('Predictions:')
for cls, info in result.items():
    print(f'  {cls:<8}  prob={info["prob"]:.4f}  pred={info["pred"]}')
